# **DocStringer**  

A Python docstring generator built by finetuning Qwen2.5-Coder with QDoRA. Given an undocumented Python function, it automatically produces a clean Google-style docstring covering the description, arguments, and return value. Built with HuggingFace and PyTorch, and with a Gradio frontend for easy use.

-------------------------------------------------------------------------------


Initial Setup:

In [1]:
# Only use the below code if on Google Colab, else comment out
from google.colab import drive
drive.mount('/content/gdrive')
!pip uninstall -y bitsandbytes accelerator peft transformers trl
!pip install -q torch==2.5.1+cu124 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q "transformers==4.47.0"
!pip install -q "peft==0.14.0"
!pip install -q "bitsandbytes==0.45.0" --extra-index-url https://vllm-wheels.pages.dev/high-efficiency
!pip install -q accelerate==1.2.1 trl==0.12.1 gradio datasets
import os
import sys
if os.path.exists('/usr/local/cuda/lib64'):
    os.environ['LD_LIBRARY_PATH'] = '/usr/local/cuda/lib64:' + os.environ.get('LD_LIBRARY_PATH', '')

Mounted at /content/gdrive
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 109.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 43.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.0 MB/s eta 0:00:00
     ━━━━━━

In [2]:
"""
If you are running this notebook locally on your own GPU setup:
- Comment out the Google Colab section above.
- Uncomment the two lines below.
- Make sure to run 'pip install -r requirements.txt' in your local terminal.

OUTPUT_DIR = "./results/qwen-docstring-qdora"
"""

'\nIf you are running this notebook locally on your own GPU setup:\n- Comment out the Google Colab section above.\n- Uncomment the two lines below.\n- Make sure to run \'pip install -r requirements.txt\' in your local terminal.\n\nOUTPUT_DIR = "./results/qwen-docstring-qdora"\n'

Imports:

In [3]:
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from peft import prepare_model_for_kbit_training,LoraConfig,PeftModel,get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device: ", device)

Using device:  cuda


In [4]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

Loading the Pre-trained Qwen model and defining the QDora Config:

In [5]:
model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = prepare_model_for_kbit_training(model)

qdora_config = LoraConfig(
    r=8,
    lora_alpha = 16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
    use_dora=True #Making it so that its QDora instead of just QLora
)

model = get_peft_model(model, qdora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

trainable params: 9,877,504 || all params: 1,553,591,808 || trainable%: 0.6358


Preparing the Dataset:

In [6]:
rawData = load_dataset("calum/the-stack-smol-python-docstrings", split="train")
print(rawData)
print("---")
# Look at a few examples
for i in range(3):
    print(f"=== Example {i+1} ===")
    print(rawData[i])
    print()

README.md:   0%|          | 0.00/669 [00:00<?, ?B/s]

data/train-00000-of-00001-ca25b7981ab6d8(…):   0%|          | 0.00/18.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/24616 [00:00<?, ? examples/s]

Dataset({
    features: ['body', 'body_hash', 'docstring', 'path', 'name', 'repository_name', 'lang', 'body_without_docstring'],
    num_rows: 24616
})
---
=== Example 1 ===
{'body': "def _canonicalize_experiment(exp):\n    'Sorts the repeated fields of an Experiment message.'\n    exp.hparam_infos.sort(key=operator.attrgetter('name'))\n    exp.metric_infos.sort(key=operator.attrgetter('name.group', 'name.tag'))\n    for hparam_info in exp.hparam_infos:\n        if hparam_info.HasField('domain_discrete'):\n            hparam_info.domain_discrete.values.sort(key=operator.attrgetter('string_value'))", 'body_hash': -8215901732217586742, 'docstring': 'Sorts the repeated fields of an Experiment message.', 'path': 'tensorboard/plugins/hparams/backend_context_test.py', 'name': '_canonicalize_experiment', 'repository_name': 'aryaman4/tensorboard', 'lang': 'python', 'body_without_docstring': "def _canonicalize_experiment(exp):\n    \n    exp.hparam_infos.sort(key=operator.attrgetter('name'))\n 

In [7]:
#Utilized Claude to generate a function for limiting the dataset to only Google-style docstrings (to exclude Sphinx style ones)
def isGoogleStyle(docstring):
    if not docstring or len(docstring.strip()) < 20:
        return False
    google_markers = ["Args:", "Returns:", "Raises:", "Example:", "Note:"]
    sphinx_markers = [":param", ":type", ":return:", ":rtype:"]
    #Exclude Sphinx style
    if any(marker in docstring for marker in sphinx_markers):
        return False

    return any(marker in docstring for marker in google_markers)

def isQualityExample(example):
    docstring = example["docstring"]
    body = example["body_without_docstring"]
    if not isGoogleStyle(docstring):
        return False
    #Function body shouldn't be too short nor too long
    if len(body.strip()) < 20:
        return False
    if len(docstring) > 1000:
        return False
    return True

filteredData = rawData.filter(isQualityExample)
print(f"Original dataset size: {len(rawData)}")
print(f"Filtered dataset size: {len(filteredData)}")
print(f"Kept {len(filteredData)/len(rawData)*100:.1f}% of examples")

Filter:   0%|          | 0/24616 [00:00<?, ? examples/s]

Original dataset size: 24616
Filtered dataset size: 2381
Kept 9.7% of examples


Transforming the dataset examples into Qwen prompts:

In [8]:
from trl import DataCollatorForCompletionOnlyLM

prompt = "You are an expert Python developer. Your task is to generate a clean, complete and accurate Google-style docstring for the given function. Your response must contain a concise description, clearly documented 'Args:' and 'Returns:' sections if applicable."
def formatPrompt(example):
  #Building the ChatML Structure
  messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Function to document:\n{example['body_without_docstring']}"},
        {"role": "assistant", "content": f"\"\"\"\n{example['docstring'].strip()}\n\"\"\""} #We need to ignore the initial and last """
  ]

  #Convert structured list into raw ChatML formatted tokens
  text = tokenizer.apply_chat_template(messages, tokenize=False)
  return {"text": text}

formattedData = filteredData.map(formatPrompt)
print(formattedData[0])


Map:   0%|          | 0/2381 [00:00<?, ? examples/s]

{'body': "def __init__(self, model_name, learning_rate=0.01, iteration=100, reconstruction_loss_weight=1.0, perceptual_loss_weight=5e-05, regularization_loss_weight=2.0, logger=None):\n    'Initializes the inverter.\\n\\n    NOTE: Only Adam optimizer is supported in the optimization process.\\n\\n    Args:\\n      model_name: Name of the model on which the inverted is based. The model\\n        should be first registered in `models/model_settings.py`.\\n      logger: Logger to record the log message.\\n      learning_rate: Learning rate for optimization. (default: 1e-2)\\n      iteration: Number of iterations for optimization. (default: 100)\\n      reconstruction_loss_weight: Weight for reconstruction loss. Should always\\n        be a positive number. (default: 1.0)\\n      perceptual_loss_weight: Weight for perceptual loss. 0 disables perceptual\\n        loss. (default: 5e-5)\\n      regularization_loss_weight: Weight for regularization loss from encoder.\\n        This is essentia

In [9]:
responseTemplate = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template=responseTemplate, tokenizer=tokenizer)
print("Prompt Formatting complete")
print(formattedData[0]['text'][:450])

Prompt Formatting complete
<|im_start|>system
You are an expert Python developer. Your task is to generate a clean, complete and accurate Google-style docstring for the given function. Your response must contain a concise description, clearly documented 'Args:' and 'Returns:' sections if applicable.<|im_end|>
<|im_start|>user
Function to document:
def __init__(self, model_name, learning_rate=0.01, iteration=100, reconstruction_loss_weight=1.0, perceptual_loss_weight=5e-05,


Setting the Training Arguments:

In [10]:
#Doing a 90% 10% train-test split
splitData = formattedData.train_test_split(test_size=0.1, seed=42)
trainData = splitData["train"]
valData = splitData["test"]

print(f"Training examples: {len(trainData)}")
print(f"Validation examples: {len(valData)}")

Training examples: 2142
Validation examples: 239


In [11]:
from transformers import TrainingArguments
from trl import SFTTrainer
#Change the below output folder path depending on if you're local or in Colab
output_dir = "/content/gdrive/MyDrive/qwen-docstring-qdora"

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=10,
    num_train_epochs=1,
    max_steps=100, #Can comment this out when running for final deploymet
    #fp16=True, # Toggle to fp16=True if on T4 GPU else bf16=True
    bf16=True,
    optim="paged_adamw_8bit",
    save_strategy="steps",
    save_steps=50,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    push_to_hub=False,
    report_to="none"
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Fine Tuning the Model:

In [12]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=trainData,
    eval_dataset=valData,
    dataset_text_field="text",
    max_seq_length=512,
    data_collator=collator,
)

print("Starting Fine-Tuning...")
trainer.train()
trainer.save_model(output_dir)
print("Fine-Tuning Complete!")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will ove

Map:   0%|          | 0/2142 [00:00<?, ? examples/s]

Map:   0%|          | 0/239 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `<|im_start|>assistant
` in the following instance: <|im_start|>system
You are an expert Python developer. Your task is to generate a clean, complete and accurate Google-style docstring for the given function. Your response must contain a concise description, clearly documented 'Args:' and 'Returns:' sections if applicable.<|im_end|>
<|im_start|>user
Function to document:
def split_train_val_forwardChaining(sequence, numInputs, numOutputs, numJumps):
    ' Returns sets to train and cross-validate a model using forward chaining technique\n    \n    Parameters:\n        sequence (array)  : Full training dataset\n        numInputs (int)   : Number of inputs X and Xcv used at each training and validation\n        numOutputs (int)  : Number of outputs y and ycv used at each training and validation\n        numJumps (int)    : Number of sequence samples to be ignored between (X,y) sets\

Starting Fine-Tuning...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
10,0.506100,0.032973
20,0.037600,0.030856
30,0.085500,0.025773
40,0.117300,0.023080
50,0.107400,0.022081
60,0.029900,0.019735
70,0.022100,0.019552
80,0.026200,0.019204
90,0.103500,0.019126
100,0.428300,0.019117


/usr/local/lib/python3.12/dist-packages/trl/trainer/utils.py:160: UserWarning: Could not find response key `<|im_start|>assistant
` in the following instance: <|im_start|>system
You are an expert Python developer. Your task is to generate a clean, complete and accurate Google-style docstring for the given function. Your response must contain a concise description, clearly documented 'Args:' and 'Returns:' sections if applicable.<|im_end|>
<|im_start|>user
Function to document:
def train_multitask_capped(model: DeepMoD, data: torch.Tensor, target: torch.Tensor, optimizer, sparsity_scheduler, test='mse', split: float=0.8, log_dir: Optional[str]=None, max_iterations: int=10000, write_iterations: int=25, **convergence_kwargs) -> None:
    '[summary]\n\n    Args:\n        model (DeepMoD): [description]\n        data (torch.Tensor): [description]\n        target (torch.Tensor): [description]\n        optimizer ([type]): [description]\n        sparsity_scheduler ([type]): [description]\n     

Fine-Tuning Complete!


Merging the QDora weights back into the base model:

In [13]:
from peft import PeftModel

print("Merging QDoRA adapters with the base model...")
# Load the saved adapters from Google Drive over your base model
inference_model = PeftModel.from_pretrained(model, output_dir)
inference_model.eval()  # Set to evaluation mode to disable dropout

def generate_docstring(raw_code: str) -> str:
    """Formats raw code into the ChatML template and streams the generated docstring."""
    messages = [
        {"role": "system", "content": prompt},
        {"role": "user", "content": f"Function to document:\n{raw_code}"}
    ]

    # add_generation_prompt=True forces the model to switch into assistant generation mode
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = inference_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,  # Keep temperature low for deterministic code structure
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # Extract only the newly generated tokens
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    ]

    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

# Test Sample
test_code = """
def process_user_records(users, target_status):
    output = []
    for user in users:
        if user.get('status') == target_status:
            output.append(user['id'])
    return output
"""

print("\n--- Testing Model Inference ---")
print(generate_docstring(test_code))

Merging QDoRA adapters with the base model...


/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:599: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_magnitude_vector.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_magnitude_vector.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_magnitude_vector.default.weight', '


--- Testing Model Inference ---
```python
"""
Process a list of user records to extract IDs based on a specified status.

Args:
    users (list): A list of dictionaries representing user records.
    target_status (str): The status to filter the users by.

Returns:
    list: A list of IDs of users who have the specified status.
"""

def process_user_records(users, target_status):
    output = []
    for user in users:
        if user.get('status') == target_status:
            output.append(user['id'])
    return output
```

This docstring provides a clear explanation of what the function does, its parameters, and what it returns. It uses Python's docstring conventions, including the `Args:` section for describing the input parameters and the `Returns:` section for describing the output.


Deploying the Frontend:

In [20]:
import gradio as gr

def ui_wrapper(raw_code):
    if not raw_code.strip():
        return "Please paste a valid Python function definition."
    try:
        return generate_docstring(raw_code)
    except Exception as e:
        return f"An error occurred during generation: {str(e)}"

with gr.Blocks(title="DocStringer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🖥️✏️ **DocStringer**")
    gr.Markdown(
        "Paste a Python function on the left. The fine-tuned Qwen2.5-Coder model "
        "will automatically generate a structured Google-style docstring on the right."
    )

    with gr.Row():
        with gr.Column():
            input_component = gr.Textbox(
                label="Your Python Code:",
                lines=12,
                placeholder="def calculate_metrics(data):\n    ..."
            )

        with gr.Column():
            output_component = gr.Textbox(
                label="Generated Google-Style Docstring:",
                lines=12,
                interactive=False
            )
    with gr.Row():
        gr.Markdown()
        submit_btn = gr.Button("Generate", variant="primary", scale=1)
        gr.Markdown()

    submit_btn.click(fn=ui_wrapper, inputs=input_component, outputs=output_component)

#Launch the server (setting share=True provides a public link valid for 72 hours)
demo.launch(share=True, debug=True)

/tmp/ipykernel_407/3581986748.py:11: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="DocStringer", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6d1528f88eadeff473.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://6d1528f88eadeff473.gradio.live


# References
-------------------------------------------------------------------------------
Tutorials and Original References
* https://www.geeksforgeeks.org/nlp/fine-tuning-large-language-models-llms-using-qlora/
* https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct
* https://huggingface.co/docs/peft/package_reference/lora
* https://claude.ai/
* https://huggingface.co/docs/trl/v0.9.6/en/sft_trainer#train-on-completions-only
* https://huggingface.co/datasets/calum/the-stack-smol-python-docstrings

Inference & Adapter Lifecycle
* Hugging Face PEFT Causal LM Inference Guide:
  https://huggingface.co/docs/peft/en/task_guides/causal_lm_adapter
* Hugging Face Generation Parameters & Strategies:
  https://huggingface.co/docs/transformers/main_classes/text_generation

UI & Deployment
* Gradio Blocks Layout Component Documentation:
  https://www.gradio.app/docs/gradio/blocks
* Hugging Face Spaces Hosting Overview:
  https://huggingface.co/docs/hub/spaces-overview